# 04. AI Search 검색 및 RAG 테스트

이미 인덱싱된 법률 데이터를 활용하여 다양한 검색 기법을 실습합니다.

| 검색 기법 | 설명 |
|----------|------|
| **키워드 검색** | BM25 전통적 텍스트 매칭 |
| **벡터 검색** | text-embedding-3-large (3072D) 의미 기반 |
| **하이브리드 검색** | 키워드 + 벡터 RRF 결합 |
| **시맨틱 재랭킹** | Azure AI Search Semantic Ranker (L2R) |
| **필터 + 벡터** | 메타데이터 필터링 후 의미 기반 검색 |
| **Multi-Index 검색** | 4개 인덱스 동시 검색 |
| **RAG** | 검색 결과 + GPT-5.4 종합 답변 |
| **Cross-Index RAG** | 여러 법률 소스 통합 RAG |

권장 순서: `02 → 03 → 04`

⚠️ **실행 위치 필수 조건**  
이 노트북은 **내부망(VNet/Private Endpoint)에 연결된 Remote VM**에서 실행해야 합니다.

## 1. 환경 설정

In [3]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType, QueryCaptionType
from openai import AzureOpenAI

SEARCH_ENDPOINT  = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT  = os.environ['AZURE_OPENAI_ENDPOINT']
GPT54_DEPLOY     = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')
EMBED_DEPLOY     = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')

from src.search.legal_indexes import LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

credential = DefaultAzureCredential()
mgr = LegalIndexManager(
    endpoint=SEARCH_ENDPOINT,
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

# OpenAI 클라이언트
token_provider = get_bearer_token_provider(
    credential,
    'https://cognitiveservices.azure.com/.default'
)
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version='2025-01-01-preview',
)

print(f"Search Endpoint : {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint : {OPENAI_ENDPOINT}")
print(f"GPT-5.4 배포명  : {GPT54_DEPLOY}")
print()

# 인덱스 통계
stats = mgr.get_all_stats()
print("인덱스 문서 개수:")
for s in stats:
    print(f"  {s['index_name']}: {s['document_count']:,}건")

Search Endpoint : https://search-ragi-dyn6dtfu.search.windows.net
OpenAI Endpoint : https://ais-ragi-dyn6dtfu.openai.azure.com/
GPT-5.4 배포명  : gpt-5.4

인덱스 문서 개수:
  prec-court-index: 18,841건
  const-court-index: 24,980건
  legis-interp-index: 8,715건
  admin-appeal-index: 6,830건


## 2. 인덱스 스키마 (실제 배포 기준 — Sweden)

### 공통 설계 원칙

| 구분 | 타입 | 설정 | 용도 |
|------|------|------|------|
| **Key 필드** | `Edm.String` | `key=true, analyzer=keyword` | 문서 고유 ID |
| **날짜 메타** | `Edm.DateTimeOffset` | `filterable, sortable, facetable, retrievable` | 날짜 범위 필터·정렬 |
| **열거형 메타** | `Edm.String` | `searchable, filterable, sortable, facetable` | 심급·유형 패싯 |
| **다중값 메타** | `Collection(Edm.String)` | `searchable, filterable, facetable` | 관련법령·주제어 (any/all 람다) |
| **본문 텍스트** | `Edm.String` | `searchable, analyzer=ko.microsoft` | 한국어 형태소 분석 |
| **장문 본문** | `Edm.String` | `searchable, analyzer=ko_safe` (filter/sort/facet 모두 false) | 32K 토큰 한도 회피 — `_text_long()` |
| **벡터 임베딩** | `Collection(Edm.Single)` | `dim=3072, vectorSearchProfile=hnsw` | 의미 기반 검색 |

**주요 설계 결정**

- **`ko.microsoft` vs `ko_safe`**: 짧은 메타·요약 필드는 표준 Microsoft 한국어 analyzer 사용. `fullText`/`reason` 같은 장문 필드는 `ko_safe` (CustomAnalyzer = `PatternReplaceCharFilter`로 한자/가나 제거 + 200자 단위 강제 분할 → `MicrosoftLanguageTokenizer(korean)` → `lowercase`)
- **장문 필드의 filter/sort/facet 비활성**: AI Search는 filterable/sortable/facetable 가 켜진 String 필드를 **단일 토큰**으로 색인하기 때문에 32 766 byte 한도를 넘으면 색인 실패. 장문은 검색만 켜고 메타 옵션은 모두 끔
- **임베딩 분리 보호**: `SplitSkill(maximumPageLength=5000)` → 첫 페이지만 `text-embedding-3-large` 호출 → 8K 토큰 한도 회피
- **`Collection(Edm.String)`**: "민법,형법" 단일 텍스트 불가 → 배열로 분리하여 람다 필터 가능
- **임베딩 필드 선택**: 전체 본문이 아닌 *요약/회답/요지* 필드만 임베딩 → 비용·정확도 균형

> **필드 attribute 표기**: `s`=searchable, `f`=filterable, `o`=sortable, `a`=facetable, `r`=retrievable, **KEY**=key

---

### 1. `prec-court-index` — 판례 (대법원·고등·지방법원)
**docs = 18,841 / storage = 522.7 MiB / vector = 108.0 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `courtName` | String | s,f,o,a,r | — | 법원명 |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `judgmentDate` | DateTimeOffset | f,o,a,r | — | 선고일자 |
| `courtLevel` | String | s,f,o,a,r | — | 심급 (1심/2심/3심) |
| `caseType` | String | s,f,o,a,r | — | 사건종류 |
| `status` | String | s,f,o,a,r | — | 진행상태 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,r | ko.microsoft | 사건명 |
| `holdings` | String | s,r | ko.microsoft | 판시사항 |
| `summary` | String | s,r | ko.microsoft | 판결요지 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 판결 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `prec-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 2. `const-court-index` — 헌법재판소 결정례
**docs = 24,980 / storage = 338.5 MiB / vector = 44.4 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `decisionDate` | DateTimeOffset | f,o,a,r | — | 결정일자 |
| `decisionType` | String | s,f,o,a,r | — | 결정유형 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `fiscalYear` | String | s,f,o,a,r | — | 귀속년도 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,f,o,a,r | ko.microsoft | 사건명 |
| `holdings` | String | s,f,o,a,r | ko.microsoft | 판시사항 |
| `summary` | String | s,f,o,a,r | ko.microsoft | 결정요지 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `const-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 3. `legis-interp-index` — 법제처 해석례
**docs = 8,715 / storage = 410.0 MiB / vector = 102.5 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `docNumber` | String | s,f,o,a,r | — | 문서번호 |
| `replyDate` | DateTimeOffset | f,o,a,r | — | 회신일자 |
| `interpType` | String | s,f,o,a,r | — | 안건종류 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `title` | String | s,r | ko.microsoft | 안건명 |
| `querySummary` | String | s,r | ko.microsoft | 질의요지 |
| `reply` | String | s,r | ko.microsoft | 회답 ← **임베딩 대상** |
| `reason` | String | s,r | **ko_safe** | 이유 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `interp-semantic` (title=`title` / content=`querySummary,reply` / keywords=`keywords,relatedLaws`)

---

### 4. `admin-appeal-index` — 행정심판 재결례
**docs = 6,830 / storage = 342.6 MiB / vector = 80.3 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `decisionDate` | DateTimeOffset | f,o,a,r | — | 재결일자 |
| `decisionType` | String | s,f,o,a,r | — | 재결유형 (인용/기각/각하) |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `committee` | String | s,f,o,a,r | — | 심판위원회 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,f,o,a,r | ko.microsoft | 사건명 |
| `order` | String | s,f,o,a,r | ko.microsoft | 주문 |
| `reasonSummary` | String | s,f,o,a,r | ko.microsoft | 이유요약 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `admin-semantic` (title=`caseName` / content=`order,reasonSummary` / keywords=`keywords,relatedLaws`)

---

### Collection(String) OData 필터 예시

```
# 특정 법령 포함
filter="relatedLaws/any(law: law eq '민법')"

# 여러 법령 중 하나
filter="relatedLaws/any(law: law eq '민법' or law eq '형법')"

# 법령 + 날짜 범위 복합
filter="relatedLaws/any(law: law eq '민법') and judgmentDate ge 2020-01-01T00:00:00Z"

# 심급 + 상태 + 날짜
filter="courtLevel eq '3심' and status eq '확정' and judgmentDate ge 2020-01-01T00:00:00Z"
```

> 4개 인덱스의 인덱싱 시간/비용/스토리지 등 종합 리포트는 [REPORT.md](../REPORT.md) 참고.


## 3. 단일 인덱스 검색 (판례)

### 3-1. 키워드 검색

In [4]:
query = "개인정보 보호 관련 판례"
results = mgr.search(PREC_INDEX, query=query, top=3, use_semantic=False)

print(f"\n[키워드 검색] '{query}' (판례, 상위 3건)\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"   법원: {r['courtName']}, 심급: {r['courtLevel']}")
    print(f"   점수: {r['score']:.4f}")
    print()


[키워드 검색] '개인정보 보호 관련 판례' (판례, 상위 3건)

1. [] 아동·청소년의성보호에관한법률위반(위계등추행)·개인정보보호법위반
   법원: , 심급: None
   점수: 52.6460

2. [] 정보통신망이용촉진및정보보호등에관한법률위반(개인정보누설등)
   법원: , 심급: None
   점수: 49.6611

3. [] 행정정보공개청구거부처분취소
   법원: , 심급: None
   점수: 45.5728



### 3-2. 하이브리드 검색 (키워드 + 벡터)

In [ ]:
query = "공직 선거 및 선거 부정방지 관련 법령 및 판례 조사해줘 "

# 쿼리 임베딩 생성
embedding_response = openai_client.embeddings.create(
    input=query,
    model=EMBED_DEPLOY,
)
embedding = embedding_response.data[0].embedding

results = mgr.search(PREC_INDEX, query=query, embedding=embedding, top=3, use_semantic=True)

print(f"\n[하이브리드 + 시맨틱 재랭킹] '{query}' (상위 3건)\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"   점수: {r['score']:.4f}")
    print(f"   판결요지: {r.get('summary', '')[:120]}...")
    print()


[하이브리드 + 시맨틱 재랭킹] '법령 개정 절차와 공포 방법은 어떻게 되나요?' (상위 3건)

1. [95누7994] 폐교처분취소
   점수: 2.1763
   판결요지: [1] 구 지방자치법(1994. 3. 16. 법률 제4741호로 개정되기 전의 것) 제15조 , 제19조 , 제35조 제1항 제1호 , 제135조 제1항 , 제2항 , 구 지방교육자치에관한법률(1995. 7. 26....

2. [95도2843] 공직선거및선거부정방지법위반
   점수: 2.1123
   판결요지: 원심은 군의회 의원선거에서 최종학력에 관하여 허위사실을 공표하게 한 범죄사실에 대하여 당시 시행 중인 공직선거및선거부정방지법 제250조 제1항 을 적용하였는데, 위 법이 상고심 계속 중 법률 제5127호(1995. ...

3. [94누5021] 건축허가신청반려처분취소
   점수: 2.0858
   판결요지: 건축법 부칙 제4조, 구 건축법시행령(1993.8.9. 대통령령 제13953호로 개정되기 전의 것) 부칙 제3조, 건축법시행규칙 부칙 제3조 등에서 지방자치단체의 조례에 위임된 사항은 위 법령 등의 시행일로부터 1년...



## 4. 필터 기반 검색 (메타데이터 필터링)

Collection(String) 필터를 사용한 정밀 검색 예시

In [4]:
# 대법원 세무 사건만 필터
print("\n[필터 검색] 판례 중 대법원(3심) 세무 사건\n")
results = mgr.search(
    PREC_INDEX,
    query="*",
    filter_expr="courtLevel eq '3심'",
    select=["caseName", "caseNumber", "courtName", "courtLevel"],
    top=5,
)
for r in results[:3]:
    print(f"[{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"  {r['courtName']} - {r['courtLevel']}\n")


[필터 검색] 판례 중 대법원(3심) 세무 사건



## 5. Multi-Index 통합 검색 (4개 인덱스 동시 검색)

4개 법률 인덱스에서 동시에 검색하여 점수 기반으로 통합 정렬합니다.

In [5]:
def cross_index_search(query: str, top_per_index: int = 3) -> list:
    """4개 인덱스 통합 검색"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=query,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 각 인덱스별 검색
    INDEX_LABELS = {
        PREC_INDEX: "판례",
        CONST_INDEX: "헌재결정례",
        INTERP_INDEX: "법제처해석례",
        ADMIN_INDEX: "행정심판재결례",
    }
    
    all_results = []
    for idx_name in [PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX]:
        results = mgr.search(idx_name, query=query, embedding=embedding, top=top_per_index)
        for r in results:
            r["_index"] = idx_name
            r["_label"] = INDEX_LABELS[idx_name]
        all_results.extend(results)
    
    # 점수 기반 정렬
    all_results.sort(key=lambda x: x.get("score", 0), reverse=True)
    return all_results


# 테스트
query = "개인정보 보호와 관련된 법률적 쟁점"
results = cross_index_search(query, top_per_index=2)

print(f"\n[Multi-Index 검색] '{query}'\n")
print(f"총 {len(results)}건 (4개 인덱스 통합, 상위 8건 표시)\n")

for i, r in enumerate(results[:8], 1):
    title = r.get("caseName") or r.get("title", "N/A")
    print(f"{i}. [{r['_label']}] (score: {r['score']:.4f})")
    print(f"   {title[:80]}")
    print()


[Multi-Index 검색] '개인정보 보호와 관련된 법률적 쟁점'

총 8건 (4개 인덱스 통합, 상위 8건 표시)

1. [판례] (score: 2.5218)
   아동·청소년의성보호에관한법률위반(위계등추행)·개인정보보호법위반

2. [헌재결정례] (score: 2.5133)
   형의 집행 및 수용자의 처우에 관한 법률 제112조 제3항 위헌확인 등

3. [판례] (score: 2.4823)
   정보공개청구거부처분취소

4. [법제처해석례] (score: 2.4158)
   공정거래위원회 - 소비자가 개인정보의 이용에 관한 동의를 철회하는 경우에도 사업자는 거래기록 및 그와 관련된 개인정보를 반드시 보존해야 하는지 

5. [헌재결정례] (score: 2.4133)
   주민등록법 제24조 제2항 위헌확인 등

6. [행정심판재결례] (score: 2.3508)
   정보공개 이의신청기각결정처분 취소청구 등

7. [행정심판재결례] (score: 2.3140)
   정보공개 이의신청 기각결정 취소청구

8. [법제처해석례] (score: 2.2646)
   민원인 - 「개인정보 보호법」 제18조제2항제3호의 “명백히 정보주체 또는 제3자의 급박한 생명, 신체, 재산의 이익을 위하여 필요하다고 인정되



## 6. RAG — GPT-5.4 기반 질의응답 (판례)

In [7]:
def rag_single_index(question: str, index_name: str, top_k: int = 3) -> str:
    """단일 인덱스 RAG"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=question,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 검색
    results = mgr.search(index_name, query=question, embedding=embedding, top=top_k)
    
    # 컨텍스트 구성
    context_parts = []
    for r in results:
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{title}] ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': '당신은 한국 법률 전문가입니다. 제공된 판례 자료를 바탕으로 사용자 질문에 정확하고 체계적으로 답변하세요. 근거 판례번호를 명시하세요.',
            },
            {
                'role': 'user',
                'content': f'## 참고 판례\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_completion_tokens=1000,
    )
    
    return response.choices[0].message.content


# 테스트
question = "손해배상 청구의 요건은 무엇인가요?"
answer = rag_single_index(question, PREC_INDEX, top_k=3)

print(f"\n[RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)



[RAG] 질문: 손해배상 청구의 요건은 무엇인가요?

손해배상 청구의 요건은, 질문에 제공된 판례들을 기준으로 보면 크게 다음과 같이 정리할 수 있습니다.

## 1. 위법한 행위 또는 불법행위가 존재해야 합니다
상대방의 행위가 법적으로 위법하여야 손해배상청구가 가능합니다.

- 예를 들어, **행정대집행의 요건을 갖추지 못했는데도 무허가 건물을 강제 철거한 경우**, 이는 위법한 공권력 행사에 해당하여 손해배상책임이 인정됩니다.
- 판례는 **성동구청장이 행정대집행 대상이 아닌 건물을 오인하여 철거한 경우 서울특별시의 손해배상책임이 있다**고 보았습니다.  
  - 근거: **손해배상청구사건(행정대집행 오단 강제철거 판례)**

## 2. 손해가 실제로 발생해야 합니다
단순히 위법행위가 있었다는 것만으로는 부족하고, 그로 인해 구체적인 손해가 발생해야 합니다.

- 예를 들어, **부당한 가처분의 취소를 위해 변호사에게 사건처리를 맡기고 보수를 지급한 경우**, 그 변호사 보수는 손해로 인정될 수 있습니다.
- 판례는 **변호사에게 지급된 상당한 정도의 보수는 손해배상으로 청구할 수 있다**고 하였습니다.  
  - 근거: **대법원 1966. 66다862**

즉, 손해는 재산적 손해뿐 아니라, 위법행위에 대응하기 위해 불가피하게 지출한 비용도 포함될 수 있습니다.

## 3. 위법행위와 손해 사이에 인과관계가 있어야 합니다
발생한 손해가 상대방의 위법행위로 인해 생긴 것임이 인정되어야 합니다.

- 위의 66다862 판례에서도, 변호사 보수는 단순한 임의 지출이 아니라 **부당한 가처분을 취소하기 위해 필요하게 된 비용**이므로 인과관계가 인정된 것입니다.
  - 근거: **대법원 66다862**

## 4. 손해배상청구권이 소멸하지 않아야 합니다
손해가 발생했더라도, 피해자가 이미 배상을 수령하여 청구권이 소멸한 경우에는 다시 청구할 수 없습니다.

- 판례에 따르면, **불법행위로 인한 피해보상금으로 공탁된 공탁금을 아무런 이의 유보 없이 수령한 

## 7. Cross-Index RAG — 여러 법률 소스 통합 답변

In [8]:
def cross_index_rag(question: str, top_per_index: int = 2) -> str:
    """Cross-Index RAG: 4개 인덱스 검색 → GPT-5.4 응답"""
    results = cross_index_search(question, top_per_index=top_per_index)
    
    context_parts = []
    for r in results[:8]:  # 상위 8건
        label = r.get("_label", "기타")
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{label}] {title} ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': (
                    '당신은 한국 법률 전문가입니다. '
                    '판례, 헌법재판소 결정례, 법제처 해석례, 행정심판 재결례 등 '
                    '다양한 법률 자료를 종합하여 사용자 질문에 정확하고 체계적으로 답변하세요.\\n'
                    '근거가 되는 사건번호와 자료 종류를 명시하세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'## 검색된 법률 자료\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_completion_tokens=1500,
    )
    
    return response.choices[0].message.content


# 테스트
question = "법령 개정 절차에 대해 설명해 주세요"
answer = cross_index_rag(question)

print(f"\n[Cross-Index RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)



[Cross-Index RAG] 질문: 법령 개정 절차에 대해 설명해 주세요

법령 개정 절차는 **무엇을 개정하느냐(법률·대통령령·총리령·부령·조례 등)**에 따라 다르지만, 큰 틀에서는 다음과 같이 이해하시면 됩니다.

---

# 1. 법령 개정의 기본 구조

일반적으로 법령 개정은 아래 순서로 진행됩니다.

1. **개정 필요성 발생**
   - 사회·경제 여건 변화
   - 상위법 개정에 따른 정비 필요
   - 위헌·위법 소지 해소
   - 제도 운영상 문제점 보완

2. **개정안 마련**
   - 소관 부처가 개정 초안을 작성
   - 이해관계자 의견수렴, 내부 검토

3. **입법예고**
   - 국민에게 개정안을 공개하고 의견을 받음
   - 행정입법(시행령·시행규칙)뿐 아니라 법률안도 통상 입법예고 절차를 거침

4. **관계기관 협의 및 심사**
   - 관련 부처 협의
   - 규제심사, 법제심사 등 진행

5. **의결 또는 의결·재가**
   - 법률안: 국무회의 심의 후 국회 제출, 국회 의결, 대통령 재가
   - 대통령령안: 국무회의 심의, 대통령 재가
   - 총리령·부령안: 필요한 심사 후 공포

6. **공포**
   - 관보 등에 게재하여 대외적으로 성립

7. **시행**
   - 공포 즉시 시행 또는 일정 유예기간 후 시행
   - 필요하면 부칙에서 경과조치, 적용례를 둠

---

# 2. 법률 개정 절차

여기서 “법률”은 국회가 제정하는 규범입니다.

## (1) 법률안 발의
- **정부제출안**: 소관 부처가 마련 → 법제처 심사 → 차관회의·국무회의 → 대통령 재가 후 국회 제출
- **의원발의안**: 국회의원이 발의

## (2) 국회 심사
- 소관 상임위원회 심사
- 필요시 법안심사소위원회 심사
- 법제사법위원회의 체계·자구 심사
- 본회의 의결

## (3) 대통령의 공포
- 국회에서 의결된 법률안은 대통령이 공포
- 공포 후 부칙이 정한 날부터 시행

---

# 3. 시행령·시행규칙 개정 절차

법

## 8. Foundry IQ — Agentic (Intelligent) Retrieval

**Foundry IQ** = Microsoft Foundry 의 지능형 검색. 내부적으로 Azure AI Search 의 두 리소스를 사용합니다:

| 리소스 | 역할 |
|---|---|
| **Knowledge Source** | 검색 대상(인덱스/Blob) 한 개를 wrapper. 인덱스당 1개 |
| **Knowledge Agent** | 다중 Knowledge Source 위에서 **질의 계획(query planning)** 을 담당 — LLM(`gpt-4o`/`gpt-4.1` 계열) |

### 동작 방식 — 단일 retrieve 호출

```
[user message + 대화 이력]
        │
        ▼
[Agent: LLM이 질문을 sub-query N개로 분해]   ← maxSubQueries 제한
        │
        ├──► Source 1 (hybrid + semantic rerank, parallel)
        ├──► Source 2  
        └──► Source 3
        │
        ▼
[reranker threshold 통과한 결과만 합쳐서 응답]
        │
        ▼
{ response, references, activity(검색 trace) }
```

### "다른 문서를 자동으로 찾아오나?" — 두 가지 시나리오

| 패턴 | 설명 | 작동 |
|---|---|---|
| **A. 단일 호출, 복합 질문** | "X사건과 그 사건이 인용한 선행 판례까지 알려줘" → agent 가 _사전에_ 여러 sub-query 계획 | ✅ |
| **B. Multi-turn 인용 추적** | 1턴: "X사건 알려줘" → 응답에 "참조: 판례 2019도1234" 포함 → 2턴: 이전 응답을 `messages` 에 넣고 "더 알려줘" → agent 가 컨텍스트의 인용번호를 보고 새 sub-query 계획 | ✅ |
| **C. Agent 가 결과를 보고 _자동으로_ 추가 호출** | 현재 API 는 단일 호출 내에서 iterative chase 를 지원하지 **않음** | ❌ |

→ **C 패턴**이 필요하면 클라이언트(예: GPT‑5.4)가 결과의 references 를 파싱해 새 retrieve 를 명시적으로 호출하는 **루프**를 직접 구현해야 합니다 (Agent‑on‑Agent 패턴, 8‑3 셀에서 시연).

> **모델 제약**: Knowledge Agent 는 `gpt-4o, gpt-4o-mini, gpt-4.1, gpt-4.1-mini, gpt-4.1-nano` 만 지원. 본 Lab 은 배포된 `gpt-4o` 사용. 응답 합성용 LLM 은 별도(`gpt-5.4`).


In [ ]:
"""8-1. Knowledge Sources + Knowledge Agent 확인

⚠️  이 리소스는 `01-infra-deployment.ipynb` 의 5c 섹션에서 이미 생성됩니다.
이 셀은 존재 여부 확인만 수행하고, 누락된 경우에만 재생성합니다 (idempotent).
"""
import json, requests
from azure.identity import DefaultAzureCredential

API_VERSION    = "2025-08-01-preview"
PLANNER_DEPLOY = "gpt-4o"
AGENT_NAME     = "legal-knowledge-agent"

INDEX_TO_SRC = {
    PREC_INDEX: ("ks-prec", "한국 법원 판례",
        "caseName,caseNumber,courtName,courtLevel,judgmentDate,relatedLaws,keywords,holdings,summary"),
    CONST_INDEX: ("ks-const", "헌법재판소 결정례",
        "caseName,caseNumber,decisionDate,decisionType,relatedLaws,keywords,holdings,summary"),
    INTERP_INDEX: ("ks-interp", "법제처 법령해석례",
        "title,docNumber,replyDate,interpType,relatedLaws,keywords,querySummary,reply"),
    ADMIN_INDEX: ("ks-admin", "행정심판 재결례",
        "caseName,caseNumber,decisionDate,decisionType,committee,relatedLaws,keywords,order,reasonSummary"),
}

cred = DefaultAzureCredential()
def _hdr():
    tok = cred.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

# 존재 여부 확인 (없으면 재생성)
need_setup = False
for _, (src_name, _, _) in INDEX_TO_SRC.items():
    url = f"{SEARCH_ENDPOINT}/knowledgesources('{src_name}')?api-version={API_VERSION}"
    if requests.get(url, headers=_hdr()).status_code != 200:
        need_setup = True; break
agent_url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')?api-version={API_VERSION}"
if requests.get(agent_url, headers=_hdr()).status_code != 200:
    need_setup = True

if not need_setup:
    print("✓ Knowledge Sources + Knowledge Agent 이미 존재 (01 노트북에서 생성됨)")
else:
    print("⚠️  누락 감지 — 재생성 시작")
    for idx_name, (src_name, desc, fields) in INDEX_TO_SRC.items():
        body = {
            "name": src_name, "kind": "searchIndex", "description": desc,
            "searchIndexParameters": {"searchIndexName": idx_name, "sourceDataSelect": fields},
        }
        url = f"{SEARCH_ENDPOINT}/knowledgesources('{src_name}')?api-version={API_VERSION}"
        r = requests.put(url, headers=_hdr(), json=body)
        print(f"  KS '{src_name}' <- {idx_name}: {r.status_code}")

    agent_body = {
        "name": AGENT_NAME,
        "description": "Korean legal corpus: 판례 / 헌재 / 법제처 / 행심",
        "models": [{"kind": "azureOpenAI", "azureOpenAIParameters": {
            "resourceUri": OPENAI_ENDPOINT.rstrip("/"),
            "deploymentId": PLANNER_DEPLOY, "modelName": PLANNER_DEPLOY,
        }}],
        "knowledgeSources": [
            {"name": s, "alwaysQuerySource": True, "includeReferences": True,
             "includeReferenceSourceData": True, "maxSubQueries": 4, "rerankerThreshold": 1.5}
            for _, (s, _, _) in INDEX_TO_SRC.items()
        ],
        "outputConfiguration": {"modality": "answerSynthesis", "includeActivity": True, "attemptFastPath": False},
        "requestLimits": {"maxRuntimeInSeconds": 60, "maxOutputSize": 16000},
        "retrievalInstructions": (
            "사용자 질문에 한국 법령·판례 자료가 필요하면 4개 소스에서 hybrid+semantic 검색을 수행한다. "
            "이전 대화 메시지에 사건번호(예: 2019도1234)나 법령명이 언급되어 있으면 그 키워드로 sub-query 를 추가 계획하라."
        ),
    }
    r = requests.put(agent_url, headers=_hdr(), json=agent_body)
    print(f"Agent '{AGENT_NAME}': {r.status_code}")


  KS 'ks-prec' <- prec-court-index: 201
  KS 'ks-const' <- const-court-index: 201
  KS 'ks-interp' <- legis-interp-index: 201
  KS 'ks-admin' <- admin-appeal-index: 201

Agent 'legal-knowledge-agent': 201
  planner = gpt-4o
  sources = ['ks-prec', 'ks-const', 'ks-interp', 'ks-admin']


In [11]:
"""8-2. retrieve helper + 단일 호출 복합 질문

복합 질문을 던져 agent 가 자동으로 여러 sub-query 를 계획하는지 activity 로 검증.
"""

def retrieve(messages: list, source_filters: dict | None = None) -> dict:
    body = {"messages": messages}
    if source_filters:
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": k, "filterAddOn": v}
            for k, v in source_filters.items()
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    r = requests.post(url, headers=_hdr(), json=body, timeout=120)
    if r.status_code >= 400:
        print(f"HTTP {r.status_code}\n{r.text[:1500]}")
        r.raise_for_status()
    return r.json()


def show_activity(resp: dict, max_refs: int = 5):
    print("\n--- ACTIVITY (query plan) ---")
    sub_q = []
    for a in resp.get("activity", []):
        t = a["type"]
        if t == "modelQueryPlanning":
            print(f"  [{a['id']}] planning  : in={a.get('inputTokens')}  out={a.get('outputTokens')}  {a.get('elapsedMs')}ms")
        elif t == "modelAnswerSynthesis":
            print(f"  [{a['id']}] synthesis : in={a.get('inputTokens')}  out={a.get('outputTokens')}  {a.get('elapsedMs')}ms")
        elif t == "searchIndex":
            args = a.get("searchIndexArguments", {})
            q = args.get("search", "")
            f = args.get("filter") or ""
            print(f"  [{a['id']}] search    : ks={a['knowledgeSourceName']:<10} hits={a['count']:>3}  q={q[:70]!r}{(' filter='+f) if f else ''}")
            sub_q.append((a["knowledgeSourceName"], q))
        elif t == "semanticReranker":
            print(f"  [{a['id']}] rerank    : in={a.get('inputTokens')}  {a.get('elapsedMs')}ms")
    print(f"\n  -> 총 sub-query: {len(sub_q)}개")

    refs = resp.get("references", [])
    print(f"\n--- REFERENCES (top {min(max_refs, len(refs))}/{len(refs)}) ---")
    for ref in refs[:max_refs]:
        sd = ref.get("sourceData", {})
        title = sd.get("caseName") or sd.get("title", "")
        case = sd.get("caseNumber") or sd.get("docNumber", "")
        print(f"  [{ref.get('rerankerScore', 0):.2f}] {case}  {title[:60]}")


def show_response(resp: dict):
    print("\n--- AGENT RESPONSE (synthesized answer) ---")
    for msg in resp.get("response", []):
        for c in msg.get("content", []):
            if c.get("type") == "text":
                print(c["text"][:1500])


# ===== 시나리오 A: 단일 호출, 복합 질문 =====
question_A = (
    "개인정보 보호와 관련해 (1) 대법원 판례, (2) 헌법재판소 결정, "
    "(3) 법제처 해석례에서 각각 핵심 사례를 알려주세요."
)
print(f"### 시나리오 A — 단일 호출 복합 질문")
print(f"질문: {question_A}\n")

resp_A = retrieve([{"role": "user", "content": [{"type": "text", "text": question_A}]}])
show_activity(resp_A)
show_response(resp_A)


### 시나리오 A — 단일 호출 복합 질문
질문: 개인정보 보호와 관련해 (1) 대법원 판례, (2) 헌법재판소 결정, (3) 법제처 해석례에서 각각 핵심 사례를 알려주세요.


--- ACTIVITY (query plan) ---
  [0] planning  : in=2186  out=139  3036ms
  [1] search    : ks=ks-prec    hits= 43  q='개인정보 보호 관련 대법원 판례 핵심 사례'
  [2] search    : ks=ks-prec    hits= 34  q='개인정보 보호 관련 헌법재판소 결정 핵심 사례'
  [3] search    : ks=ks-prec    hits= 41  q='개인정보 보호 관련 법제처 해석례 핵심 사례'
  [4] search    : ks=ks-const   hits= 48  q='개인정보 보호 관련 대법원 판례 핵심 사례'
  [5] search    : ks=ks-const   hits= 50  q='개인정보 보호 관련 헌법재판소 결정 핵심 사례'
  [6] search    : ks=ks-const   hits= 49  q='개인정보 보호 관련 법제처 해석례 핵심 사례'
  [7] search    : ks=ks-interp  hits= 48  q='개인정보 보호 관련 대법원 판례 핵심 사례'
  [8] search    : ks=ks-interp  hits= 47  q='개인정보 보호 관련 헌법재판소 결정 핵심 사례'
  [9] search    : ks=ks-interp  hits= 49  q='개인정보 보호 관련 법제처 해석례 핵심 사례'
  [10] search    : ks=ks-admin   hits= 46  q='개인정보 보호 관련 대법원 판례 핵심 사례'
  [11] search    : ks=ks-admin   hits= 48  q='개인정보 보호 관련 헌법재판소 결정 핵심 사례'
  [12] search    : ks=ks-admin   hits= 44  

In [12]:
"""8-3. 시나리오 B — Multi-turn 인용 추적

이전 turn 의 agent 응답을 conversation history 로 다음 retrieve 호출에 포함하면,
agent 의 query planner 는 응답에 언급된 사건번호/법령을 보고 새 sub-query 를 자동 계획.
"""
import re

# Turn 1: 손해배상 청구의 요건 — 응답에서 인용 판례번호가 언급되도록 유도
turn1_q = "민법상 손해배상 청구의 요건을 정리해 주세요. 핵심 판례 사건번호도 함께 알려주세요."
print(f"### 시나리오 B-Turn 1\n질문: {turn1_q}\n")

resp_B1 = retrieve([{"role": "user", "content": [{"type": "text", "text": turn1_q}]}])
show_activity(resp_B1, max_refs=3)

# Turn 1 응답 텍스트 추출
turn1_answer = ""
for msg in resp_B1.get("response", []):
    for c in msg.get("content", []):
        if c.get("type") == "text":
            turn1_answer = c["text"]
print(f"\n--- Turn 1 응답 (앞 600자) ---\n{turn1_answer[:600]}\n...")

# 응답 안에서 사건번호 패턴 (예: 2019다1234, 2020도5678) 추출
cited = re.findall(r"\d{4}[가-힣]{1,2}\d{2,6}", turn1_answer)
print(f"\n응답에서 추출한 사건번호 (중복제거): {sorted(set(cited))[:5]}")

# Turn 2: 이전 응답을 conversation history 로 넘기고 인용 판례 상세 요청
turn2_q = "위에서 언급된 판례들 중 가장 중요한 1-2건을 골라 사실관계와 법원 판단을 자세히 설명해 주세요."
messages_B2 = [
    {"role": "user",      "content": [{"type": "text", "text": turn1_q}]},
    {"role": "assistant", "content": [{"type": "text", "text": turn1_answer}]},
    {"role": "user",      "content": [{"type": "text", "text": turn2_q}]},
]
print(f"\n### 시나리오 B-Turn 2 (with history)\n질문: {turn2_q}\n")
resp_B2 = retrieve(messages_B2)
show_activity(resp_B2, max_refs=5)

# Turn 2 sub-query 가 turn 1 의 인용번호를 포함하는지 검사
turn2_queries = [
    a["searchIndexArguments"]["search"]
    for a in resp_B2.get("activity", [])
    if a["type"] == "searchIndex"
]
hits = [q for q in turn2_queries if any(c in q for c in set(cited))]
print(f"\n→ Turn 2 sub-query 중 turn 1 인용번호 포함: {len(hits)}/{len(turn2_queries)}")
for q in hits[:3]:
    print(f"   • {q}")


### 시나리오 B-Turn 1
질문: 민법상 손해배상 청구의 요건을 정리해 주세요. 핵심 판례 사건번호도 함께 알려주세요.


--- ACTIVITY (query plan) ---
  [0] planning  : in=2170  out=103  5226ms
  [1] search    : ks=ks-prec    hits= 50  q='민법상 손해배상 청구 요건'
  [2] search    : ks=ks-prec    hits= 50  q='민법 손해배상 관련 판례 사건번호'
  [3] search    : ks=ks-interp  hits= 37  q='민법상 손해배상 청구 요건'
  [4] search    : ks=ks-interp  hits= 32  q='민법 손해배상 관련 판례 사건번호'
  [5] search    : ks=ks-const   hits= 48  q='민법상 손해배상 청구 요건'
  [6] search    : ks=ks-const   hits= 50  q='민법 손해배상 관련 판례 사건번호'
  [7] search    : ks=ks-admin   hits= 39  q='민법상 손해배상 청구 요건'
  [8] search    : ks=ks-admin   hits= 43  q='민법 손해배상 관련 판례 사건번호'
  [9] rerank    : in=206553  Nonems
  [10] synthesis : in=18038  out=159  8371ms

  -> 총 sub-query: 8개

--- REFERENCES (top 3/33) ---
  [2.82]   손해배상청구
  [2.75] 2016다210474  손해배상(기)
  [2.71]   손해배상(기)

--- Turn 1 응답 (앞 600자) ---
민법상 손해배상 청구의 요건은 다음과 같습니다: 1) 가해자의 고의 또는 과실, 2) 위법한 행위, 3) 손해의 발생, 4) 가해 행위와 손해 간의 인과관계. 관련 판례로는 "채무불이행으로 인한 손해배상의 예정이 있는 

In [13]:
"""8-4. 시나리오 C — Agent-on-Agent 자동 인용 추적 루프

Knowledge Agent 자체는 단일 호출 안에서 결과를 보고 추가 호출하지 않음.
"자동으로 인용 문서를 따라가는" 동작을 만들려면 클라이언트 루프 필요:
  1. 첫 retrieve → references 의 sourceData 에서 relatedLaws / 사건번호 추출
  2. 그 키워드로 새 retrieve 호출
  3. 두 결과 합쳐 최종 답변 합성

이 셀은 그 패턴을 시연 (max 2 hop).
"""
def extract_followup_terms(refs: list, top_n: int = 5) -> list[str]:
    """references 의 relatedLaws / keywords 에서 후속 검색용 키워드 추출"""
    terms: dict[str, int] = {}
    for ref in refs:
        sd = ref.get("sourceData", {})
        for fld in ("relatedLaws", "keywords"):
            val = sd.get(fld)
            if isinstance(val, list):
                for v in val:
                    if v and 2 <= len(v) <= 30:
                        terms[v] = terms.get(v, 0) + 1
        # 사건번호도 후속 단서
        cn = sd.get("caseNumber") or sd.get("docNumber")
        if cn:
            terms[cn] = terms.get(cn, 0) + 1
    # 빈도 상위
    return [k for k, _ in sorted(terms.items(), key=lambda x: -x[1])[:top_n]]


def agentic_chase(initial_q: str, hops: int = 2) -> dict:
    history = [{"role": "user", "content": [{"type": "text", "text": initial_q}]}]
    all_refs: list = []
    hop_log = []

    for hop in range(hops):
        print(f"\n=== HOP {hop+1} ===")
        resp = retrieve(history)
        show_activity(resp, max_refs=3)

        # 응답 누적
        for msg in resp.get("response", []):
            history.append(msg)
        all_refs.extend(resp.get("references", []))
        hop_log.append({
            "hop": hop+1,
            "subqueries": [a.get("searchIndexArguments", {}).get("search")
                           for a in resp.get("activity", []) if a["type"] == "searchIndex"],
        })

        if hop == hops - 1:
            break
        # 다음 hop 키워드 추출 후 새 user message 추가
        followups = extract_followup_terms(resp.get("references", []), top_n=5)
        print(f"\n→ HOP {hop+1} references 에서 추출한 후속 키워드: {followups}")
        if not followups:
            print("  (후속 키워드 없음 — 루프 종료)")
            break
        next_q = (
            f"앞서 찾은 자료들에서 다음 키워드/법령/사건번호와 관련된 추가 자료를 찾아 더 자세히 설명해주세요: "
            + ", ".join(followups)
        )
        history.append({"role": "user", "content": [{"type": "text", "text": next_q}]})

    return {"history": history, "references": all_refs, "hops": hop_log}


# 실행
print("### 시나리오 C — 클라이언트 루프 자동 인용 추적")
result = agentic_chase("개인정보 보호법 위반에 대한 형사 처벌의 핵심 판례를 알려주세요", hops=2)

print("\n\n=== 최종 요약 ===")
for h in result["hops"]:
    print(f"HOP {h['hop']}: {len(h['subqueries'])}개 sub-query")
    for q in h["subqueries"][:5]:
        print(f"   • {(q or '')[:80]}")
print(f"누적 references: {len(result['references'])}건")


### 시나리오 C — 클라이언트 루프 자동 인용 추적

=== HOP 1 ===

--- ACTIVITY (query plan) ---
  [0] planning  : in=2162  out=115  2784ms
  [1] search    : ks=ks-prec    hits= 34  q='개인정보 보호법 위반 형사 처벌 판례'
  [2] search    : ks=ks-prec    hits= 40  q='개인정보 보호법 위반 관련 주요 판례'
  [3] search    : ks=ks-prec    hits= 41  q='개인정보 보호법 위반 형사 사건 판례'
  [4] search    : ks=ks-const   hits= 46  q='개인정보 보호법 위반 형사 처벌 판례'
  [5] search    : ks=ks-const   hits= 47  q='개인정보 보호법 위반 관련 주요 판례'
  [6] search    : ks=ks-const   hits= 50  q='개인정보 보호법 위반 형사 사건 판례'
  [7] search    : ks=ks-interp  hits= 50  q='개인정보 보호법 위반 형사 처벌 판례'
  [8] search    : ks=ks-interp  hits= 50  q='개인정보 보호법 위반 관련 주요 판례'
  [9] search    : ks=ks-interp  hits= 48  q='개인정보 보호법 위반 형사 사건 판례'
  [10] search    : ks=ks-admin   hits= 40  q='개인정보 보호법 위반 형사 처벌 판례'
  [11] search    : ks=ks-admin   hits= 44  q='개인정보 보호법 위반 관련 주요 판례'
  [12] search    : ks=ks-admin   hits= 43  q='개인정보 보호법 위반 형사 사건 판례'
  [13] rerank    : in=445945  Nonems
  [14] synthesis : in=18164  out=100

### 8-5. 시나리오 D — LLM-as-Judge 자동 루프 (진짜 Agentic)

시나리오 C 는 우리가 키워드 추출 규칙을 짜서 hop 을 강제로 돌렸음.
시나리오 D 는 **Knowledge Agent 의 `retrieve` 를 OpenAI function tool 로 등록** 하고
`gpt-4o` 가 직접:
1. 사용자 질문을 보고 → tool 호출 (Knowledge Agent retrieve)
2. 결과 봄 → "이걸로 답할 수 있나?" **자체 판단**
3. 부족하면 다른 query 로 또 호출, 충분하면 final answer 생성
4. 사용자는 그냥 한 번 호출하고 결과만 받음 (replan 개입 0)

**왜 Foundry Agent Service 를 안 쓰는가:**
- Foundry Agent Service 는 새 리소스/SDK 필요 (`azure-ai-projects`)
- 동일한 "자체 판단 루프" 를 기존 Azure OpenAI `gpt-4o` + function calling 으로 얻을 수 있음
- 판단 주체는 둘 다 LLM — 차이는 thread 상태 저장 위치 (서버 vs 클라이언트) 뿐
- Lab 목적엔 투명한 클라이언트 루프가 학습/디버깅에 더 유리

**아키텍처:**
```
User question
    ↓
gpt-4o (chat.completions, tools=[knowledge_agent_retrieve])
    │
    ├─ tool_call → retrieve(query="...", source="ks-prec")
    │     ↓ (Knowledge Agent: 4 sub-query 분해 + rerank + synthesize)
    │     answer + references
    │
    ├─ "부족" → 다른 source 로 또 tool_call (자동)
    │
    └─ "충분" → finish_reason=stop → 최종 답변
```


In [ ]:
"""8-5. 시나리오 D — LLM-as-Judge 자동 tool 루프 (자기-완결 셀)

핵심: openai_client.chat.completions.create(tools=[...]) 를 우리(클라이언트)가 while 로 돌린다.
서버는 한 turn 만 plan 하고 tool_calls 를 돌려줄 뿐, 다음 turn 호출 책임은 우리에게 있음.
"""
import json, time
from requests.exceptions import HTTPError

PLANNER_DEPLOY_D = "gpt-4o"   # tool 호출 가능한 모델
COMPLEX_QUESTION = (
    "전세사기 피해를 입은 임차인이 임대인의 보증금 미반환으로 민사·행정 구제를 모색하고 있습니다. "
    "다음을 종합 분석해 주세요:\n"
    " (1) 임대보증금 반환청구권 행사와 우선변제권에 관한 대법원 핵심 판례 (사건번호 인용),\n"
    " (2) 주택임대차보호법상 임차인 보호 범위에 대한 헌법재판소 결정 (위헌·헌법불합치 여부 포함),\n"
    " (3) 임차권등기명령 신청 절차에 대한 법제처 법령해석례,\n"
    " (4) 전세사기 피해자 지원 관련 행정심판 재결례.\n"
    "각 source 별로 최소 1건 이상 인용하고, 마지막에 임차인이 취해야 할 단계별 권리구제 절차를 정리해 주세요."
)


# ── tool 본체: Knowledge Agent retrieve 호출 (429 backoff 포함) ──
def tool_knowledge_agent_retrieve(query: str, source_filter: str = "all") -> str:
    """[Tool] 한국 법률 자료 4종(ks-prec/ks-const/ks-interp/ks-admin)에서 자연어 질의로 검색·요약."""
    body: dict = {"messages": [{"role": "user", "content": [{"type": "text", "text": query}]}]}
    if source_filter and source_filter != "all":
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": source_filter, "filterAddOn": ""}
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    delay = 5
    for attempt in range(4):
        r = requests.post(url, headers=_hdr(), json=body, timeout=120)
        if r.status_code == 429:
            print(f"   ⏳ 429 — {delay}s 대기 ({attempt+1}/4)")
            time.sleep(delay); delay *= 2
            continue
        r.raise_for_status()
        resp = r.json()
        break
    else:
        raise RuntimeError("retrieve 429 retries exhausted")

    answer = "\n".join(
        c.get("text", "") for m in resp.get("response", []) for c in m.get("content", [])
        if c.get("type") == "text"
    ).strip() or "(no answer)"
    refs = []
    for ref in resp.get("references", [])[:8]:
        sd = ref.get("sourceData", {})
        title = sd.get("caseName") or sd.get("title") or sd.get("docNumber") or "?"
        case_no = sd.get("caseNumber") or sd.get("docNumber", "")
        refs.append(f"- [{case_no}] {title}")
    sub_q = sum(1 for a in resp.get("activity", []) if a.get("type") == "searchIndex")
    return json.dumps({"answer": answer[:2000], "references": refs, "sub_queries_executed": sub_q},
                      ensure_ascii=False)


# ── tool schema (function calling) ──
TOOL_DEFS = [{
    "type": "function",
    "function": {
        "name": "tool_knowledge_agent_retrieve",
        "description": "한국 법률 자료(판례/헌재/법제처/행심) 통합 검색. source_filter 로 특정 소스만 호출 가능.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "자연어 검색 쿼리"},
                "source_filter": {
                    "type": "string",
                    "enum": ["all", "ks-prec", "ks-const", "ks-interp", "ks-admin"],
                    "description": "특정 소스 한정 (기본 'all')",
                },
            },
            "required": ["query"],
        },
    },
}]


# ── ★ 핵심: 클라이언트 측 while 루프 ★ ──
def autonomous_agent(question: str, max_iters: int = 8) -> dict:
    messages = [
        {"role": "system", "content": (
            "너는 한국 법률 리서치 어시스턴트다. tool_knowledge_agent_retrieve 를 자유롭게 호출해 "
            "자료를 모은 뒤, 충분하다고 판단되면 사건번호를 인용한 종합 답변을 한국어로 생성하라. "
            "필요한 경우 여러 source(ks-prec, ks-const, ks-interp, ks-admin) 를 호출해 자료를 보강하되, "
            "질문에 무관한 source 까지 강제로 호출할 필요는 없다."
        )},
        {"role": "user", "content": question},
    ]
    tool_calls_total = 0
    for it in range(max_iters):
        resp = openai_client.chat.completions.create(
            model=PLANNER_DEPLOY_D,
            messages=messages,
            tools=TOOL_DEFS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            print(f"\n🏁 iter={it+1} finish=stop  (총 tool 호출 {tool_calls_total}회)")
            break
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"  [iter {it+1}] tool: {tc.function.name}  source={args.get('source_filter','all')}  q={args.get('query','')[:60]!r}")
            try:
                result = tool_knowledge_agent_retrieve(**args)
            except Exception as e:
                result = json.dumps({"error": str(e)}, ensure_ascii=False)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            tool_calls_total += 1
    print("\n=== 최종 답변 (D — 클라이언트 루프) ===")
    print(messages[-1].get("content", "")[:2000])
    return {"messages": messages, "tool_calls": tool_calls_total}


print("### 시나리오 D — LLM-as-Judge 자동 tool 루프\n")
print(f"질문:\n{COMPLEX_QUESTION}\n")
result_D = autonomous_agent(COMPLEX_QUESTION)

### 시나리오 D — LLM-as-Judge 자동 tool 루프

질문:
전세사기 피해를 입은 임차인이 임대인의 보증금 미반환으로 민사·행정 구제를 모색하고 있습니다. 다음을 종합 분석해 주세요:
 (1) 임대보증금 반환청구권 행사와 우선변제권에 관한 대법원 핵심 판례 (사건번호 인용),
 (2) 주택임대차보호법상 임차인 보호 범위에 대한 헌법재판소 결정 (위헌·헌법불합치 여부 포함),
 (3) 임차권등기명령 신청 절차에 대한 법제처 법령해석례,
 (4) 전세사기 피해자 지원 관련 행정심판 재결례.
각 source 별로 최소 1건 이상 인용하고, 마지막에 임차인이 취해야 할 단계별 권리구제 절차를 정리해 주세요.


🔧 HOP 1 tool call: query='임대보증금 반환청구권 행사와 우선변제권 관련 대법원 판례' source=ks-prec

🔧 HOP 1 tool call: query='주택임대차보호법상 임차인 보호 범위에 대한 헌법재판소 위헌·헌법불합치 여부' source=ks-const
HTTP 429
{"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '429' (TooManyRequests). Too Many Requests"}}
   ⏳ 429 — 5s 대기 후 재시도 (1/4)
HTTP 429
{"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '429' (TooManyRequests). Too Many Requests"}}
   ⏳ 429 — 10s 대기 후 재시도 (2/4)

🔧 HOP 1 tool call: query='임차권등기명령 신청 및 법적 절차 관련 법제처 법령 해석' source=ks-interp
HTTP 429
{"error":{"code":"","messag

### 8-6. 시나리오 E — Azure AI Foundry Agent Service (서버 측 자율 루프)

시나리오 D 와 같은 자체-판단 루프지만, **상태 관리·tool 실행·LLM 호출이 모두 Azure 서버 측에서** 일어남.
클라이언트는 thread 만들고 message 보내고 run 만들고 결과만 받음.

| 항목 | Scenario D (Chat Completions + tools) | Scenario E (Foundry Agent Service) |
|---|---|---|
| LLM 자체 판단 루프 | ✅ | ✅ |
| Thread/대화 상태 저장 | 클라이언트 messages 리스트 | Azure 서버 측 |
| Tool 등록 위치 | 매 요청 `tools=[...]` | Agent 정의에 영구 등록 |
| Built-in tools | 없음 (직접 wrap) | **`AzureAISearchTool`, `FileSearchTool`, `CodeInterpreter`, `BingGroundingTool`** 등 즉시 사용 |
| 모니터링/Trace | 직접 구현 | Foundry Portal 에 자동 기록 |
| Multi-agent 오케스트레이션 | 직접 | `connected_agents` 로 agent-to-agent |

**선결 인프라 (한 번만 수행):**
```bash
# 1. Bicep 패치 — AIServices 계정에 allowProjectManagement=true + Foundry project 생성
az deployment group create \
  --resource-group rg-rag-indexing-lab-swc \
  --template-file infra/sweden/patches/add-foundry-project.bicep \
  --parameters accountName=ais-ragi-<suffix> location=swedencentral

# 2. 사용자에게 project RBAC 부여 (Azure AI User)
PROJECT_ID=$(az cognitiveservices account project show \
  --account-name ais-ragi-<suffix> -g rg-rag-indexing-lab-swc -n proj-ragi-default \
  --query id -o tsv)
az role assignment create --assignee <user-object-id> \
  --role "Azure AI User" --scope "$PROJECT_ID"

# 3. SDK 설치 (이미 pyproject.toml 에 추가됨)
uv pip install -e .
```

**환경변수:** `.env` 에 `FOUNDRY_PROJECT_ENDPOINT=https://ais-ragi-<suffix>.services.ai.azure.com/api/projects/proj-ragi-default` 추가


In [ ]:
"""8-6. 시나리오 E — Foundry Agent Service: 서버 측 자율 루프 (자기-완결 셀)"""
import os, json, time, requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

SEARCH_ENDPOINT = os.environ.get("SEARCH_ENDPOINT") or os.environ["AZURE_SEARCH_ENDPOINT"]
AGENT_NAME      = os.getenv("KNOWLEDGE_AGENT_NAME", "legal-knowledge-agent")
API_VERSION     = "2025-08-01-preview"
FOUNDRY_PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

cred = DefaultAzureCredential()
_search_token = cred.get_token("https://search.azure.com/.default").token


def _retrieve(messages: list, source_filter: str | None = None) -> dict:
    body = {"messages": messages}
    if source_filter and source_filter != "all":
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": source_filter, "filterAddOn": ""}
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    headers = {"Authorization": f"Bearer {_search_token}", "Content-Type": "application/json"}
    delay = 5
    for attempt in range(4):
        r = requests.post(url, headers=headers, json=body, timeout=120)
        if r.status_code == 429:
            print(f"   ⏳ Search 429 — {delay}s 대기 ({attempt+1}/4)")
            time.sleep(delay); delay *= 2
            continue
        if r.status_code >= 400:
            print(f"   HTTP {r.status_code}: {r.text[:500]}")
        r.raise_for_status()
        return r.json()
    raise RuntimeError("retrieve 429 retries exhausted")


def knowledge_agent_retrieve(query: str, source_filter: str = "all") -> str:
    """[Tool] 한국 법률 자료 4종(ks-prec/ks-const/ks-interp/ks-admin)에서 자연어 질의로 검색·요약."""
    msgs = [{"role": "user", "content": [{"type": "text", "text": query}]}]
    resp = _retrieve(msgs, source_filter=source_filter)
    answer_parts = []
    for m in resp.get("response", []):
        for c in m.get("content", []):
            if c.get("type") == "text":
                answer_parts.append(c.get("text", ""))
    answer = "\n".join(answer_parts).strip() or "(no answer)"
    refs_summary = []
    for ref in resp.get("references", [])[:8]:
        sd = ref.get("sourceData", {})
        title = sd.get("caseName") or sd.get("title") or sd.get("docNumber") or "?"
        case_no = sd.get("caseNumber") or sd.get("docNumber", "")
        refs_summary.append(f"- [{case_no}] {title}")
    activity_count = sum(1 for a in resp.get("activity", []) if a.get("type") == "searchIndex")
    return json.dumps({
        "answer": answer[:2000],
        "references": refs_summary,
        "sub_queries_executed": activity_count,
    }, ensure_ascii=False)


COMPLEX_QUESTION = (
    "전세사기 피해를 입은 임차인이 임대인의 보증금 미반환으로 민사·행정 구제를 모색하고 있습니다. "
    "다음을 종합 분석해 주세요:\n"
    " (1) 임대보증금 반환청구권 행사와 우선변제권에 관한 대법원 핵심 판례 (사건번호 인용),\n"
    " (2) 주택임대차보호법상 임차인 보호 범위에 대한 헌법재판소 결정 (위헌·헌법불합치 여부 포함),\n"
    " (3) 임차권등기명령 신청 절차에 대한 법제처 법령해석례,\n"
    " (4) 전세사기 피해자 지원 관련 행정심판 재결례.\n"
    "각 source 별로 최소 1건 이상 인용하고, 마지막에 임차인이 취해야 할 단계별 권리구제 절차를 정리해 주세요."
)


# ── Foundry Agent Service (azure-ai-agents 1.1+) ──
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import FunctionTool, ToolSet, ListSortOrder

# 신형 SDK: AgentsClient 를 project endpoint 로 직접 생성
ag_client = AgentsClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=cred)

functions = FunctionTool(functions={knowledge_agent_retrieve})
toolset = ToolSet()
toolset.add(functions)
ag_client.enable_auto_function_calls(toolset)  # ← 클라이언트 측 자동 tool 실행 활성화

FOUNDRY_AGENT_NAME = "legal-foundry-agent"
existing = [a for a in ag_client.list_agents() if a.name == FOUNDRY_AGENT_NAME]
if existing:
    agent = existing[0]
    print(f"♻️  재사용: {agent.id}")
else:
    agent = ag_client.create_agent(
        model="gpt-4o",
        name=FOUNDRY_AGENT_NAME,
        instructions=(
            "너는 한국 법률 리서치 어시스턴트다. "
            "knowledge_agent_retrieve tool 을 자유롭게 호출해 자료를 모은 뒤, "
            "충분하다고 판단되면 사건번호를 인용한 종합 답변을 한국어로 생성하라. "
            "필요한 경우 여러 source(ks-prec, ks-const, ks-interp, ks-admin) 를 호출해 자료를 보강하라. "
            "다만 질문에 무관한 source 까지 강제로 호출할 필요는 없다."
        ),
        toolset=toolset,
    )
    print(f"✨ 생성: {agent.id}")

# create_thread_and_process_run = 서버에서 plan→tool→plan→... 자동 진행, 한 번 호출로 종료
print(f"\n📨 자율 루프 실행 중 (서버측 자동 LLM↔tool 반복)...\n")
t0 = time.time()
run = ag_client.create_thread_and_process_run(
    agent_id=agent.id,
    thread={"messages": [{"role": "user", "content": COMPLEX_QUESTION}]},
    toolset=toolset,
)
elapsed = time.time() - t0
print(f"🏁 status={run.status}, elapsed={elapsed:.1f}s")
if run.status == "failed":
    print(f"❌ {run.last_error}")

# 메시지 + 사용량
print(f"\n📊 usage: prompt_tokens={run.usage.prompt_tokens}, completion_tokens={run.usage.completion_tokens}")

messages = list(ag_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.ASCENDING))
print(f"\n💬 thread 총 메시지 {len(messages)}개")

# Run steps 로 실제 tool 호출 횟수 확인 (서버측 기록)
steps = list(ag_client.run_steps.list(thread_id=run.thread_id, run_id=run.id))
tool_steps = [s for s in steps if s.type == "tool_calls"]
print(f"🔧 서버측 tool 호출 단계: {len(tool_steps)}개")
for i, s in enumerate(tool_steps, 1):
    if hasattr(s.step_details, "tool_calls"):
        for tc in s.step_details.tool_calls:
            if hasattr(tc, "function"):
                args_preview = (tc.function.arguments or "")[:120]
                print(f"  step {i}: {tc.function.name} args={args_preview}")

print("\n=== 최종 답변 (Foundry 서버 자율 결과) ===")
for m in messages:
    if m.role == "assistant":
        for c in m.content:
            if hasattr(c, "text"):
                print(c.text.value)
                print("---")


✨ 생성: asst_cFQK3Bv99pLORCbqlnxTjQBo

📨 자율 루프 실행 중 (서버측 자동 LLM↔tool 반복)...



Error executing function 'knowledge_agent_retrieve': 403 Client Error: Forbidden for url: https://search-ragi-dyn6dtfu.search.windows.net/agents('legal-knowledge-agent')/retrieve?api-version=2025-08-01-preview


   HTTP 403: {"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '403' (Forbidden). Public access is disabled. Please configure private endpoint."}}


Error executing function 'knowledge_agent_retrieve': 403 Client Error: Forbidden for url: https://search-ragi-dyn6dtfu.search.windows.net/agents('legal-knowledge-agent')/retrieve?api-version=2025-08-01-preview


   HTTP 403: {"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '403' (Forbidden). Public access is disabled. Please configure private endpoint."}}


Error executing function 'knowledge_agent_retrieve': 403 Client Error: Forbidden for url: https://search-ragi-dyn6dtfu.search.windows.net/agents('legal-knowledge-agent')/retrieve?api-version=2025-08-01-preview


   HTTP 403: {"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '403' (Forbidden). Public access is disabled. Please configure private endpoint."}}


Error executing function 'knowledge_agent_retrieve': 403 Client Error: Forbidden for url: https://search-ragi-dyn6dtfu.search.windows.net/agents('legal-knowledge-agent')/retrieve?api-version=2025-08-01-preview
Tool outputs contain errors - retrying


   HTTP 403: {"error":{"code":"","message":"Could not complete model action. The model endpoint returned status code '403' (Forbidden). Public access is disabled. Please configure private endpoint."}}
🏁 status=RunStatus.COMPLETED, elapsed=15.5s

📊 usage: prompt_tokens=1642, completion_tokens=231

💬 thread 총 메시지 2개
🔧 서버측 tool 호출 단계: 1개
  step 1: knowledge_agent_retrieve args={"query": "임대보증금 반환청구권 행사와 우선변제권에 관한 대법원 핵심 판례", "source_filter": "ks-prec"}
  step 1: knowledge_agent_retrieve args={"query": "주택임대차보호법상 임차인 보호 범위에 대한 헌법재판소 결정, 위헌 여부 포함", "source_filter": "ks-const"}
  step 1: knowledge_agent_retrieve args={"query": "임차권등기명령 신청 절차에 대한 법제처 법령해석례", "source_filter": "ks-interp"}
  step 1: knowledge_agent_retrieve args={"query": "전세사기 피해자 지원 관련 행정심판 재결례", "source_filter": "ks-admin"}

=== 최종 답변 (Foundry 서버 자율 결과) ===
법률 자료 검색 요청이 작동하지 않았습니다. 이를 해결한 후 다시 요청해 주시거나, 다른 질문이 있다면 말씀해 주세요!
---


## 8-7. 시나리오 F — 인용 판례 자동 추적 (Citation Chase)

### 🎯 목표
"메인 판례를 검색"한 뒤, **본문에 인용된 다른 사건번호**를 LLM 이 스스로 발견하고 그 인용 판례까지 후속 조회하는 진정한 멀티-홉 reasoning 데모.

### 🔍 데이터 분석으로 찾은 시드
실제 인덱스를 스캔한 결과 **성수대교 붕괴 사건 (97도1740, 대법원 1997.11.28)** 을 시드로 선정:

| 항목 | 값 |
|---|---|
| 메인 사건 | `97도1740` 업무상과실치사·치상·일반교통방해·자동차추락 |
| 본문 인용 사건번호 | `94도35`, `96도1231` (둘 다 인덱스에 존재 ✓) |
| 추적 가능 깊이 | 2-hop (메인 → 인용 판례) |

### 🧠 에이전트 동작 패턴
1. **Hop 1**: 사용자 질문을 `knowledge_agent_retrieve` 로 검색 → 답변/references 에 `94도35`, `96도1231` 등 사건번호 등장
2. **자율 판단**: instructions 에 따라 LLM 이 "이 사건번호들을 추가로 조회하면 답변 품질이 향상됨" 을 인식
3. **Hop 2**: 추출한 사건번호로 `knowledge_agent_retrieve` 재호출 → 인용 판례의 판시사항·요지 획득
4. **종합**: 메인 판례 + 인용 판례를 통합한 답변 생성

### ⚠️ 스키마 제약
현재 `prec-court-index` 에는 명시적 `referencedCases` 필드가 없으므로, 인용 정보는 `fullText` 본문에 자연어로만 존재합니다. 따라서 LLM 의 패턴 인식(`\d{2,4}[가-힣]{1,2}\d{2,6}`) 능력에 의존합니다 — 이것이 이 시나리오의 핵심 어려움입니다.


In [ ]:
"""8-7. 시나리오 F — Citation Chase

전제: 셀 8-6 (Scenario E) 가 먼저 실행되어 다음이 정의되어 있어야 합니다:
  - ag_client (AgentsClient)
  - knowledge_agent_retrieve (function tool)
  - cred, ListSortOrder
"""
import time, re, json
from azure.ai.agents.models import FunctionTool, ToolSet, ListSortOrder

# ── Citation-chase 전용 에이전트 (instructions 만 다름) ──
CHASE_AGENT_NAME = "legal-foundry-agent-chase"
CHASE_INSTRUCTIONS = (
    "너는 한국 법률 판례 분석 전문가다. knowledge_agent_retrieve tool 을 사용해 "
    "ks-prec (판례) source 를 조회한다.\n\n"
    "★ 중요한 멀티-홉 규칙 ★\n"
    "1) 첫 번째 검색 결과의 답변·references 본문을 자세히 읽어, "
    "**다른 사건번호**(예: '94도35', '2011다86720', '81누67' 등 "
    "정규식 패턴 \\d{2,4}[가-힣]{1,2}\\d{2,6} 형태)가 인용되어 있는지 확인하라.\n"
    "2) 인용된 사건번호가 사용자의 질문과 직접 관련 있어 보이면, "
    "**그 사건번호를 query 로 한 후속 knowledge_agent_retrieve 호출** 을 추가로 수행하라. "
    "(source_filter='ks-prec' 로 지정)\n"
    "3) 인용 판례를 추적했음에도 인덱스에 없거나 무관하면 그 사실을 답변에 명시하라.\n"
    "4) 무한 루프를 피하기 위해 최대 4회까지만 추가 조회하고, 동일 사건번호는 재조회하지 마라.\n\n"
    "최종 답변에는 (a) 메인 판례 요지, (b) 추적한 인용 판례별 요지, "
    "(c) 인용 관계가 본 사안에 미치는 의미를 한국어로 정리하라. "
    "모든 사건번호는 [97도1740] 형식으로 인용하라."
)

functions = FunctionTool(functions={knowledge_agent_retrieve})  # noqa: F821 (셀 8-6 정의)
chase_toolset = ToolSet()
chase_toolset.add(functions)
ag_client.enable_auto_function_calls(chase_toolset)  # noqa: F821

existing = [a for a in ag_client.list_agents() if a.name == CHASE_AGENT_NAME]  # noqa: F821
if existing:
    chase_agent = existing[0]
    # instructions 가 바뀌었을 수 있으니 update
    chase_agent = ag_client.update_agent(  # noqa: F821
        agent_id=chase_agent.id, instructions=CHASE_INSTRUCTIONS, toolset=chase_toolset
    )
    print(f"♻️  재사용+갱신: {chase_agent.id}")
else:
    chase_agent = ag_client.create_agent(  # noqa: F821
        model="gpt-4o",
        name=CHASE_AGENT_NAME,
        instructions=CHASE_INSTRUCTIONS,
        toolset=chase_toolset,
    )
    print(f"✨ 생성: {chase_agent.id}")

# ── 추적이 필요한 질문 (성수대교 사건을 자연스럽게 유도) ──
CHASE_QUESTION = (
    "1997년 대법원이 선고한 성수대교 붕괴 사건(업무상과실치사·치상 등) 판결의 핵심 법리를 알려주고, "
    "그 판결이 본문에서 인용한 선행 판례들(다른 사건번호로 표기된 판례)이 있다면 "
    "그 인용 판례들의 판시사항도 함께 조회해서 본 사건과의 관계를 정리해 주세요."
)

print(f"\n📨 Citation-chase 자율 루프 실행 중...\n")
t0 = time.time()
run = ag_client.create_thread_and_process_run(  # noqa: F821
    agent_id=chase_agent.id,
    thread={"messages": [{"role": "user", "content": CHASE_QUESTION}]},
    toolset=chase_toolset,
)
elapsed = time.time() - t0
print(f"🏁 status={run.status}, elapsed={elapsed:.1f}s")
if run.status == "failed":
    print(f"❌ {run.last_error}")

print(f"\n📊 usage: prompt_tokens={run.usage.prompt_tokens}, completion_tokens={run.usage.completion_tokens}")

# ── tool 호출 chain 분석 ──
steps = list(ag_client.run_steps.list(thread_id=run.thread_id, run_id=run.id))  # noqa: F821
tool_steps = [s for s in steps if s.type == "tool_calls"]
print(f"\n🔧 서버측 tool 호출 단계: {len(tool_steps)}개  (≥2 면 chase 성공)")

queries_issued = []
for i, s in enumerate(tool_steps, 1):
    if hasattr(s.step_details, "tool_calls"):
        for tc in s.step_details.tool_calls:
            if hasattr(tc, "function"):
                try:
                    args = json.loads(tc.function.arguments or "{}")
                    q = args.get("query", "")[:140]
                except Exception:
                    q = (tc.function.arguments or "")[:140]
                queries_issued.append(q)
                print(f"  step {i}: {tc.function.name}(query='{q}')")

# ── 인용 추적 성공 여부 평가 ──
CITE_RE = re.compile(r"\d{2,4}[가-힣]{1,2}\d{2,6}")
seed_case = "97도1740"
follow_up_cases = set()
for q in queries_issued[1:]:
    follow_up_cases.update(CITE_RE.findall(q))
follow_up_cases.discard(seed_case)

print(f"\n🔎 후속 query 에서 추출된 사건번호: {sorted(follow_up_cases) or '(없음)'}")
if follow_up_cases:
    print(f"✅ Citation chase 성공: {len(follow_up_cases)} 개 인용 판례를 자율적으로 추적함")
else:
    print(f"⚠️  Citation chase 미발생: 메인 검색 결과만으로 답변 생성 (instructions/모델 한계 가능)")

# ── 최종 답변 ──
messages = list(ag_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.ASCENDING))  # noqa: F821
print("\n=== 최종 답변 (Citation Chase 결과) ===")
for m in messages:
    if m.role == "assistant":
        for c in m.content:
            if hasattr(c, "text"):
                print(c.text.value)
                print("---")
